In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
!pip install -q pysentimiento pandas


In [ ]:
import pandas as pd
import re

df = pd.read_csv("comentarios_experimento_youtube.csv")
df.head()

texto_col = "text"


In [ ]:
def limpiar_texto(t):
    t = str(t)
    t = re.sub(r"http\S+", " ", t)      # URLs
    t = re.sub(r"@\w+", " ", t)        # menciones (por si las hay)
    t = re.sub(r"#", " ", t)           # #
    t = re.sub(r"\s+", " ", t).strip()
    return t

df = df.dropna(subset=[texto_col])
df = df.drop_duplicates(subset=[texto_col])

df["texto_limpio"] = df[texto_col].apply(limpiar_texto)
df[["texto_limpio"]].head()


In [ ]:
from pysentimiento import create_analyzer

# Modelo de sentimiento en español (pos/neg/neu) basado en BETO
analyzer = create_analyzer(task="sentiment", lang="es")


In [ ]:
def analizar_sentimiento(texto):
    res = analyzer.predict(texto)
    return pd.Series({
        "label": res.output,                 # 'POS', 'NEG', 'NEU'
        "proba_pos": res.probas.get("POS", 0.0),
        "proba_neg": res.probas.get("NEG", 0.0),
        "proba_neu": res.probas.get("NEU", 0.0),
    })

resultados = df["texto_limpio"].apply(analizar_sentimiento)

df_beto = pd.concat([df, resultados], axis=1)
df_beto.head()


In [ ]:
df_beto["label"].value_counts()


In [ ]:
print("🔴Ejemplos NEGATIVOS")
df_beto[df_beto["label"] == "NEG"][["texto_limpio", "proba_neg"]].head(10)

print("🟢Ejemplos POSITIVOS")
df_beto[df_beto["label"] == "POS"][["texto_limpio", "proba_pos"]].head(10)

print("⚪️Ejemplos NEUTROS")
df_beto[df_beto["label"] == "NEU"][["texto_limpio", "proba_neu"]].head(10)


In [ ]:
df_beto.to_csv("comentarios_experimento_youtube_beto.csv", index=False)

from google.colab import files
files.download("comentarios_experimento_youtube_beto.csv")


Notebook de la descarga de los datos: https://colab.research.google.com/drive/1dzZmis_AmT-xwanq0JG82cVV1JzpTmGo?usp=sharing